# Will the gate see anything?

A CyberGate kit can serve perfectly and gate **nothing**. `is_baseline_prefill` decides which
prefill is the one real observation of the user's turn; `publish_decision` drops every prefill it
rejects. If it rejects the render the server actually observes, the gate scores nothing, writes no
record, dumps no activation row and logs nothing, while the model answers normally. `check_model`
cannot catch that -- the dimensions agree.

The reverse costs just as much. Every request runs several prefills: the middleware's own
observation probe, then a `<think>`-injected route or a pass-through. If the filter accepts more
than one of them, the later one **overwrites the published decision** and, with a dump running,
appends a second unlabelled fp16 row -- which a later fit consumes as data.

This notebook measures both, for every render CyberGate can emit plus the one the client's own
chat request produces. **No GPU, no vllm, no weights** -- the family classes are pure Python and
the tokenizer is ~15 MB. It costs about three minutes and it is worth running before any serve,
any harvest, and after any change to a family's template attributes.

**Bound.** One single-turn conversation with no system message, one checkpoint's tokenizer. It
says nothing about multi-turn renders, tool turns, or prefix-cached prefills, each of which takes
a different branch of the predicate. It also does not prove the hook calls the predicate with
these arguments -- that needs a live serve and a row count. See the last markdown cell.


In [ ]:
REPO_URL = "https://github.com/kncshw/cybergate.git"  # @param {type:"string"}
BRANCH   = "main"  # @param {type:"string"}

import os, subprocess, sys, pathlib, importlib

if not pathlib.Path("/content/cybergate").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL,
                    "/content/cybergate"], check=True)
os.chdir("/content/cybergate")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

# `--no-deps` on purpose: nothing here imports torch or vllm, and installing vllm costs 5-15
# minutes and several GB. If this notebook ever starts needing them, that is the bug.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)

# Colab keeps /content on sys.path, so `import cybergate` would find the CLONE DIRECTORY as a PEP
# 420 namespace package, succeed, and fail one attribute later. src/ first makes the real package
# win without a kernel restart.
SRC = "/content/cybergate/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in [k for k in sys.modules if k == "cybergate" or k.startswith("cybergate.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import cybergate
from cybergate.families import ALL_FAMILIES
print("cybergate from", cybergate.__file__)
print("families     ", [f.name for f in ALL_FAMILIES])


In [ ]:
KIT = "qwen38"  # @param ["qwen38", "gemma4"]

import json, pathlib
from huggingface_hub import snapshot_download
from cybergate.families import family_for_model_id

MAN   = json.loads(pathlib.Path("/content/cybergate/kits", KIT, "MANIFEST.json").read_text())
MODEL = MAN["model_ids"][0]
fam   = family_for_model_id(MODEL)

# Tokenizer only. `*.jinja` matters: recent transformers ships `chat_template.jinja` as its own
# file, and without it `apply_chat_template` does not render what the server renders.
LOCAL = snapshot_download(MODEL, allow_patterns=["*.json", "*.txt", "*.model", "*.jinja"])

print(f"kit     {KIT}")
print(f"model   {MODEL}")
print(f"family  {fam.name}  verified_live={fam.verified_live}")
print(f"tokenizer at {LOCAL}")
print("\nIf the download 403s, the repo is gated:  from huggingface_hub import login; login()")


In [ ]:
from transformers import AutoTokenizer

tok  = AutoTokenizer.from_pretrained(LOCAL)
msgs = [{"role": "user", "content": "ping"}]

#: Every mode `ModelFamily._thought_block` can render. Exactly ONE of them -- `baseline`, the mode
#: `middleware._probe_observations` hardcodes -- is the observation prefill and must publish. The
#: rest append server-authored text after the user's turn; publishing one would overwrite the real
#: decision with a score read off CyberGate's own injected prompt.
PASS2_MODES = ("preclosed", "preclosed_no_opener", "closed_loop_step1", "closed_loop_step1b",
               "closed_loop_step2", "closed_loop_cl2_step2", "sandwich_step1", "sandwich_step2")


def encode(text):
    """One encoder for every row. Taking ids from `apply_chat_template(tokenize=True)` instead is
    NOT comparable: depending on the transformers version it returns a list, a `BatchEncoding` or a
    batched `[[...]]`, and the wrong shape finds no markers, degrades to `full_sequence_fallback`
    -- which this predicate publishes unconditionally -- so a shape bug reads as a real finding."""
    return list(tok(text, add_special_tokens=False).input_ids)


def shape(ids):
    """(u1_start, suffix_len, span_method, does_it_publish) for a token id sequence."""
    spans, method = fam.locate_all_user_spans(ids)
    u1, suf = spans[0][0], len(ids) - spans[-1][1]
    return u1, suf, method, bool(fam.is_baseline_prefill(u1, suf, method, 12))


def row(label, ids, must_publish):
    u1, suf, method, ok = shape(ids)
    print(f"  {label:<38} u1_start={u1:<4} suffix_len={suf:<5} "
          f"{'PUBLISHES' if ok else 'dropped':<10} {method}"
          f"{'   <== NOT WHAT THIS MUST DO' if ok != must_publish else ''}")
    return (label, u1, suf, method, ok, must_publish)


accepted = getattr(fam, "baseline_suffix_lens", None) or (fam.default_suf, 9)
print(f"{fam.name}: the predicate accepts suffix lengths {accepted}, and the Qwen arm")
print(f"{'':8}additionally requires u1_start > 3\n")

results = []
print("  --- renders CyberGate itself emits (serving/prompts.py -> build_raw_prompt) ---")
for mode in ("baseline",) + PASS2_MODES:
    try:
        # `include_bos` must match what `middleware._probe_observations` passes, which is
        # `family.supports_bos` -- for a BOS family, rendering without it reports a
        # `u1_start` one lower than the one the probe actually produces.
        text = fam.build_raw_prompt(msgs, mode=mode, include_bos=fam.supports_bos)
    except ValueError as exc:
        print(f"  {mode:<38} build_raw_prompt rejected this mode: {exc}")
        continue
    results.append(row(f"{mode}  {'(THE OBSERVATION)' if mode == 'baseline' else ''}",
                       encode(text), must_publish=(mode == "baseline")))

print("\n  --- renders the CLIENT's own chat request produces (vllm + apply_chat_template) ---")
print("      These are answer-generation prefills, not observations. Each must be DROPPED.")
for label, kw in (("default", {}), ("enable_thinking=True", {"enable_thinking": True}),
                  ("enable_thinking=False", {"enable_thinking": False})):
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **kw)
    except TypeError:
        print(f"  client {label:<31} template does not accept this kwarg")
        continue
    results.append(row(f"client {label}", encode(text), must_publish=False))
    if label == "default":
        print(f"      render = {text!r}")

probe = next(r for r in results if r[0].startswith("baseline"))
print(f"\n  probe render = (u1_start={probe[1]}, suffix_len={probe[2]})")
if fam.name == "qwen38":
    print(f"  a decision record measured live on 2026-11-02 carried (14, 9)  ->  "
          f"{'MATCH' if (probe[1], probe[2]) == (14, 9) else 'MISMATCH: the observed render has changed'}")

wrong = [r for r in results if r[4] != r[5]]
print()
if not wrong:
    print("  OK: exactly the observation prefill publishes. The gate sees the user's turn once.")
for label, u1, suf, method, ok, must in wrong:
    if must and not ok:
        print(f"  [DEAD GATE]      {label.strip()} is the observation and it is DROPPED. "
              f"This kit gates nothing: no record, no dump row, no log line.")
    elif method == "full_sequence_fallback":
        print(f"  [NO MARKERS]     {label.strip()} matched none of this family's turn markers, so "
              f"span location fell back to the whole sequence -- and the predicate publishes that "
              f"branch UNCONDITIONALLY, pooling the score over the system prompt and the "
              f"generation prompt too. Check the printed render above: if it does open a user "
              f"turn with the expected tokens, suspect this notebook before the kit.")
    else:
        print(f"  [DOUBLE PUBLISH] {label.strip()} is not an observation and it PUBLISHES "
              f"(u1_start={u1}, suffix_len={suf}). It will overwrite the real decision, and "
              f"with CYBERGATE_DUMP_ACTS_DIR set it appends a second unlabelled row.")


## How to read it

`OK` means exactly one render publishes, and it is the one `middleware._probe_observations` sends.
That is the whole invariant.

**`[DEAD GATE]`** -- the kit is serving and scoring nothing. Measure the probe's `suffix_len` from
the table and make the family's predicate accept it. Do not widen the predicate until you know
which render is the observation; widening is the other failure.

**`[DOUBLE PUBLISH]`** -- the predicate is too loose. Note *which* guard is doing the work: a
client render with `u1_start=3` is dropped by the Qwen arm's `u1_start > 3` test and not by its
suffix length at all, so a template that injects a default system turn and one that does not are
two different situations at the same `suffix_len`. The table prints both numbers for that reason.

### What this does not establish

That the hook calls the predicate with these arguments. The renders here are measured, the
plumbing is not. One live request settles it: serve with `CYBERGATE_DUMP_ACTS_DIR` and
`CYBERGATE_DUMP_ACTS_LAYERS=<gate layer>` set, send a single prompt, and count rows --

```python
import pathlib, time
b = pathlib.Path("/content/rowcheck/L40.bin"); w = fam.hidden_size * 2
n0 = b.stat().st_size // w if b.is_file() else 0
# ... send exactly one request ...
time.sleep(2); print(f"+{b.stat().st_size // w - n0} rows")
```

`+1` is correct. `+2` means two prefills were published for one request. `scripts/fit/
harvest_vllm_acts.py` makes that assertion after every single request of a harvest, which is why
row `i` can be trusted to be request `i` -- and why a harvest aborts rather than quietly producing
a dump whose labels are off by one from the moment the filter drifted.


In [ ]:
# Optional: every registered family, same check, one cell. Each needs its own tokenizer
# download (~15 MB), and a gated repo will raise -- gemma4 is gated on the Hub.
from cybergate.families import ALL_FAMILIES

for f in ALL_FAMILIES:
    print("=" * 94)
    print(f"{f.name}  (gate L{f.expected_gate[0]} {f.expected_gate[1]}, ref L{f.ref_layer})")
    try:
        man = json.loads(pathlib.Path("/content/cybergate/kits", f.kit_name,
                                      "MANIFEST.json").read_text())
        local = snapshot_download(man["model_ids"][0],
                                  allow_patterns=["*.json", "*.txt", "*.model", "*.jinja"])
        t = AutoTokenizer.from_pretrained(local)
    except Exception as exc:
        print(f"  skipped: {type(exc).__name__}: {str(exc)[:150]}")
        continue
    base = f.build_raw_prompt(msgs, mode="baseline", include_bos=f.supports_bos)
    ids = list(t(base, add_special_tokens=False).input_ids)
    spans, method = f.locate_all_user_spans(ids)
    u1, suf = spans[0][0], len(ids) - spans[-1][1]
    ok = f.is_baseline_prefill(u1, suf, method, 12)
    print(f"  probe  u1_start={u1:<4} suffix_len={suf:<5} "
          f"{'PUBLISHES' if ok else 'DROPPED -- GATES NOTHING':<26} {method}")
    cl = list(t.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True))
    spans, method = f.locate_all_user_spans(cl)
    u1c, sufc = spans[0][0], len(cl) - spans[-1][1]
    okc = f.is_baseline_prefill(u1c, sufc, method, 12)
    print(f"  client u1_start={u1c:<4} suffix_len={sufc:<5} "
          f"{'PUBLISHES -- DOUBLE PUBLISH' if okc else 'dropped':<26} {method}")
